# Notebook 03 — Cloud/Shadow Masking and NDVI
**Phase:** create/apply a cloud and shadow mask using the Sentinel-2 Level-2A Scene Classification Layer (SCL).

Project scene: Sentinel-2B Level-2A, 9 October 2026, tile T43PGS. B04/B08 are 10 m; SCL is commonly 20 m and must be aligned to the target raster grid before pixel-by-pixel masking.

This notebook documents the workflow. Paths and the SCL exclusion rules must be checked against the original scripts before producing final results.

## 1. Set paths and find the inputs

In [ ]:
from pathlib import Path
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling

R10M_DIR = Path(r"C:\PATH\TO\PRODUCT.SAFE\GRANULE\YOUR_GRANULE\IMG_DATA\R10m")
R20M_DIR = Path(r"C:\PATH\TO\PRODUCT.SAFE\GRANULE\YOUR_GRANULE\IMG_DATA\R20m")
def find_one(folder, pattern):
    found = list(folder.glob(pattern)) if folder.exists() else []
    return found[0] if found else None
red_path = find_one(R10M_DIR, "*B04_10m.jp2")
nir_path = find_one(R10M_DIR, "*B08_10m.jp2")
scl_path = find_one(R20M_DIR, "*SCL_20m.jp2")
print("B04:", red_path, "\nB08:", nir_path, "\nSCL:", scl_path)

## 2. Check metadata and grid alignment
B04 and B08 should have matching dimensions, transform, and CRS. SCL often differs because it is 20 m. Categorical SCL values must be resampled with nearest-neighbour, not bilinear/cubic interpolation.

In [ ]:
for label, path in [("B04",red_path),("B08",nir_path),("SCL",scl_path)]:
    if path is None:
        print(label, "NOT FOUND — correct the folder path.")
    else:
        with rasterio.open(path) as ds:
            print(label, "shape", (ds.height,ds.width), "CRS", ds.crs,
                  "resolution", ds.res, "NoData", ds.nodata)

## 3. SCL classes and exclusion set
Common SCL classes include 3 (cloud shadow), 8 (medium-probability cloud), 9 (high-probability cloud), 10 (cirrus), 11 (snow/ice), 1 (saturated/defective), and 0 (no data). The exclusion set is a method choice; verify it against the original processing script.

In [ ]:
EXCLUDED_SCL_CLASSES = {0, 1, 3, 8, 9, 10, 11}
print(sorted(EXCLUDED_SCL_CLASSES))

## 4. Apply an already aligned SCL raster to an NDVI raster
This example expects `SCL_ALIGNED` to already be on the exact NDVI grid. If not, align it first using `Resampling.nearest`. Never silently combine mismatched grids.

In [ ]:
NDVI_INPUT = Path(r"C:\PATH\TO\Anantapur_NDVI_20261009.tif")
SCL_ALIGNED = Path(r"C:\PATH\TO\SCL_aligned_to_NDVI_grid.tif")
NDVI_OUTPUT = Path(r"C:\PATH\TO\Anantapur_NDVI_Masked_20261009.tif")
if NDVI_INPUT.exists() and SCL_ALIGNED.exists():
    with rasterio.open(NDVI_INPUT) as nds, rasterio.open(SCL_ALIGNED) as sds:
        if nds.shape != sds.shape or nds.transform != sds.transform or nds.crs != sds.crs:
            raise ValueError("NDVI and SCL grids differ; align SCL with nearest-neighbour first.")
        a = nds.read(1).astype("float32")
        scl = sds.read(1)
        valid = np.isfinite(a)
        if nds.nodata is not None: valid &= a != nds.nodata
        keep = valid & ~np.isin(scl, list(EXCLUDED_SCL_CLASSES))
        out = np.where(keep, a, -9999).astype("float32")
        profile = nds.profile.copy()
        profile.update(dtype="float32", nodata=-9999, compress="deflate")
    with rasterio.open(NDVI_OUTPUT, "w", **profile) as dst: dst.write(out, 1)
    print("Saved:", NDVI_OUTPUT, "retained pixels:", int(keep.sum()))
else:
    print("Skipped — set both paths to existing files.")

## Troubleshooting / errors to record
- **File not found:** check extracted SAFE path and R10m/R20m folders.
- **Grid mismatch:** align SCL to the target raster using nearest-neighbour.
- **All pixels masked:** inspect actual SCL class values, exclusion set, and alignment.
- **Clouds remain:** SCL masking is not perfect; inspect output visually.
- **Huge GeoTIFF:** retain locally and regenerate; do not push files over GitHub's standard 100 MiB limit.

Only log errors that actually occurred. The supplied notebooks did not contain executed error outputs, so this is a troubleshooting guide, not a claim that each error happened.

## Phase outcome
Compare before/after valid-pixel counts, inspect the mask in QGIS, and document the exact SCL classes excluded. The final mask depends on correct alignment and justified class choices.